# CompIL6

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.CompIL6)

class CompIL6(LinearReferenceClock):
    pass



In [3]:
model = pya.models.CompIL6()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "compil6"
model.metadata["data_type"] = "DNA methylation"  # Paper: A DNA-methylation predictor of IL-6 was developed from CpG methylation levels.
model.metadata["species"] = "Homo sapiens"  # Paper: The training cohort comprised Lothian Birth Cohort 1936 participants.
model.metadata["year"] = 2021
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Stevenson, A. J., et al. “Creating and Validating a DNA Methylation-Based Proxy for Interleukin-6.” The Journals of Gerontology: Series A 76(12): 2284–2292 (2021)."
model.metadata["doi"] = "https://doi.org/10.1093/gerona/glab046"
model.metadata["notes"] = "Thirty-five-CpG whole-blood proxy score for persistent IL-6-related inflammatory burden, fitted against covariate-adjusted normalized plasma IL-6."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: DNA extracted from whole blood supplied the methylation predictors.
model.metadata["predicts"] = ["interleukin-6"]  # Paper: The 35-CpG weighted sum was generated as a DNAm proxy for IL-6.
model.metadata["training_target"] = ["interleukin-6"]  # Paper: IL-6 NPX was inverse normalized and adjusted for age, sex, genetic principal components, and Olink plate; standardized residuals entered penalized regression.
model.metadata["unit"] = ["unitless"]  # Paper: The proxy is computed by multiplying beta values by coefficients and summing into one score.
model.metadata["model_type"] = "elastic net regression"  # Paper: The elastic net model used glmnet with 12-fold cross-validation and alpha 0.5.
model.metadata["platform"] = ["Illumina 450K"]  # Paper: Training whole-blood DNA was analyzed on the Illumina 450K BeadChip.
model.metadata["population"] = "older adults"  # Paper: The predictor was developed in 875 LBC1936 older adults with mean age 70.
model.metadata["journal"] = "The Journals of Gerontology: Series A"
model.metadata["last_author"] = "Riccardo E. Marioni"
model.metadata["n_features"] = 35
model.metadata["citations"] = 55
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
supplementary_url = "https://raw.githubusercontent.com/Duzhaozhen/OmniAge/c10fbe8cb92957520fbff1d55ae1def0691252e5/OmniAgePy/src/omniage/data/CompIL6.csv"
supplementary_file_name = "coefficients.csv"
os.system(f"curl -sL -o {supplementary_file_name} {supplementary_url}")

0

## Load features

In [6]:
df = pd.read_csv('coefficients.csv')
if str(df.columns[0]).startswith('Unnamed'):
    df = df.iloc[:, 1:]
mask = df['probe'].astype(str).str.lower().isin(['intercept', '(intercept)'])
intercept_value = float(df.loc[mask, 'coef'].iloc[0]) if mask.any() else 0.0
coef_df = df.loc[~mask].reset_index(drop=True)
model.features = coef_df['probe'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['coef'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Stevenson, Anna J., et al. "Creating and validating a DNA '
             'methylation-based proxy for interleukin-6." The Journals of '
             'Gerontology: Series A 76.12 (2021): 2284-2292.',
 'clock_name': 'compil6',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1093/gerona/glab046',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2021}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg26144437', 'cg03076319', 'cg05575921', 'cg21368161', 'cg04642923', 'cg21123519', 'cg04583842', 'cg16357224', 'cg24455236', 'cg25323809', 'cg03885055', 'cg17412005', 'cg19638572', 'cg04921989', 'cg14965639', 'cg04381957', 'cg20789595', 'cg24935598', 'cg26230601', 'cg16508480'

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: coefficients.csv
